# Cuaderno de archivos csv 
## Objetivo: Generar a partir de los archivos json de los hospitales y las rutas una versión simplificada en formato csv con los datos necesarios que ocupa el proyecto.

In [7]:
#Imports necesarios
import geopandas as gpd
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

In [9]:
#cargamos el archivo JSON de la info con los hospitales
gdf = gpd.read_file("hospitales.geojson")

print(gdf.head())

                  id                @id addr:city addr:housenumber  \
0  relation/11632168  relation/11632168      None             None   
1       way/27804337       way/27804337    México              116   
2     node/319436637     node/319436637      None             None   
3     node/739035810     node/739035810      None             None   
4      way/883666969      way/883666969      None    120 y Canario   

  addr:postcode             addr:street   amenity building check_date  \
0          None                    None  hospital     None        NaT   
1         01120                 Sur 136  hospital     None        NaT   
2          None                    None  hospital     None        NaT   
3          None                    None  hospital     None        NaT   
4         01140  General Felipe Ángeles  hospital     None        NaT   

       created_by  ...                                           operator  \
0            None  ...               Instituto Mexicano del Seg

In [11]:
#Observamos los atributos para quitar los que no hagan falta y poblar los que requieran.
gdf.columns

Index(['id', '@id', 'addr:city', 'addr:housenumber', 'addr:postcode',
       'addr:street', 'amenity', 'building', 'check_date', 'created_by',
       'description', 'emergency', 'healthcare', 'healthcare:speciality',
       'internet_access', 'name', 'note', 'opening_hours', 'operator',
       'operator:short', 'operator:type', 'operator:wikidata', 'phone', 'type',
       'website', 'wheelchair', '@geometry', 'geometry'],
      dtype='object')

In [13]:
#Generamos un csv con la información que queremos del archivo json original
hospitales = gdf[[
    "id",
    "name",
    "healthcare:speciality",
    "emergency",
    "operator:short",
    "geometry"
]].copy()


hospitales["lat"] = hospitales.geometry.y
hospitales["lon"] = hospitales.geometry.x


hospitales = hospitales.rename(columns={
    "id": "id_osm",
    "name": "nombre",
    "healthcare:speciality": "especialidades_osm",
    "operator:short": "institucion",
    "emergency": "emergencias"
})

hospitales = hospitales[[
    "id_osm",
    "nombre",
    "lat",
    "lon",
    "especialidades_osm",
    "emergencias",
    "institucion",
]]

hospitales

,id_osm,nombre,lat,lon,especialidades_osm,emergencias,institucion
0,relation/11632168,Complejo de hospitales IMSS,19.336820,-99.200272,None,yes,IMSS
1,way/27804337,Centro Médico ABC,19.400212,-99.203998,general;paediatrics;gynaecology;emergency;surg...,yes,None
2,node/319436637,Hospital San Angel Inn,19.340353,-99.199841,None,None,None
3,node/739035810,Clínica Hospital 8 IMSS,19.336612,-99.200969,None,None,IMSS
4,way/883666969,Hospital General Dr. Fernando Quiroz Gutiérrez...,19.394387,-99.194223,None,None,ISSSTE
5,way/986068924,Centro de Atención a la Salud T-III Jalalpa,19.371641,-99.238921,None,None,None
6,way/1364990384,None,19.360337,-99.176509,None,None,None
7,way/1365216619,"Hospital Regional ""Lic. Adolfo López Mateos""",19.358356,-99.172605,None,yes,ISSSTE
8,way/1365266055,"Hospital Regional ""Lic. Adolfo López Mateos""",19.358713,-99.173006,None,None,ISSSTE
9,node/2324519237,Hospital General Enrique Cabrera,19.361589,-99.224313,None,yes,None


In [15]:
#Cambios artificiales hechos con el fin de probar la correcta ejecución del proyecto 
# NO SON DATOS REALES: FUERON ALTERACIONES HECHAS AL JSON ORIGINAL

hospitales["emergencias"] = hospitales["emergencias"].fillna("yes")

hospitales.loc[0, "especialidades_osm"] = "general;trauma;urgencias"
hospitales.loc[1, "especialidades_osm"] = "general;cardiologia;pediatria;ginecologia;emergencia;cirugia;cardiologia;oncologia;cardiologia;radiologia;intensivo;interno" 
#en tupla 1 el valor original era:general;paediatrics;gynaecology;emergency;surgery;cardiology;oncology;radiology;intensive;internal
hospitales.loc[2, "especialidades_osm"] = "pediatria;urgencias;cirugia"
hospitales.loc[3, "especialidades_osm"] = "general;trauma;neurologia;urgencias"
hospitales.loc[4, "especialidades_osm"] = "general;cardiologia;radiologia;intensivo"
hospitales.loc[5, "especialidades_osm"] = "pediatria;ginecologia;urgencias"
hospitales.loc[6, "especialidades_osm"] = "general;cirugia;oncologia;radiologia"
hospitales.loc[7, "especialidades_osm"] = "trauma;ortopedia;urgencias;cirugia"
hospitales.loc[8, "especialidades_osm"] = "general;neurologia;cardiologia;intensivo"
hospitales.loc[9, "especialidades_osm"] = "pediatria;general;urgencias;interno"
hospitales.loc[10, "especialidades_osm"] = "general;ginecologia;maternidad;cirugia"
hospitales.loc[11, "especialidades_osm"] = "trauma;urgencias;radiologia;intensivo"
hospitales.loc[12, "especialidades_osm"] = "general;cardiologia;oncologia;neurologia"

In [17]:
hospitales.to_csv("hospitales_limpio.csv", index=False)

In [19]:
#creamos un csv de la capacidad de cada hspital DE FORMA ARTIFICIAL (Nada de esto es real)
#En este archivo se asignarán camas disponibles y ocupadas por cada especialidad del hospital
import random
import pandas as pd

filas = []

for i, row in hospitales.iterrows():

    especialidades = row["especialidades_osm"].split(";")

    for esp in especialidades:

        filas.append({
            "id_hospital": row["id_osm"],
            "hospital": row["nombre"],
            "especialidad": esp.strip(),

            # Datos simulados
            "camas_totales": random.randint(1, 15),

            "camas_ocupadas": random.randint(0, 10)
        })

capacidades = pd.DataFrame(filas)

capacidades

,id_hospital,hospital,especialidad,camas_totales,camas_ocupadas
0,relation/11632168,Complejo de hospitales IMSS,general,12,6
1,relation/11632168,Complejo de hospitales IMSS,trauma,6,1
2,relation/11632168,Complejo de hospitales IMSS,urgencias,3,6
3,way/27804337,Centro Médico ABC,general,3,8
4,way/27804337,Centro Médico ABC,cardiologia,1,0
5,way/27804337,Centro Médico ABC,pediatria,8,3
6,way/27804337,Centro Médico ABC,ginecologia,13,3
7,way/27804337,Centro Médico ABC,emergencia,8,3
8,way/27804337,Centro Médico ABC,cirugia,10,4
9,way/27804337,Centro Médico ABC,cardiologia,14,6


In [21]:
capacidades.to_csv("capacidades.csv", index=False)


In [23]:
#migramos la información de los hospitales como nodos
nodos = hospitales[["id_osm", "nombre", "lat", "lon"]].copy()

nodos = nodos.rename(columns={
    "id_osm": "id_nodo",
    "nombre": "nombre_nodo"
})

nodos["tipo"] = "hospital"

nodos.to_csv("nodos.csv", index=False)
nodos.to_excel("nodos.xlsx", index=False)

nodos.head()

,id_nodo,nombre_nodo,lat,lon,tipo
0,relation/11632168,Complejo de hospitales IMSS,19.336820,-99.200272,hospital
1,way/27804337,Centro Médico ABC,19.400212,-99.203998,hospital
2,node/319436637,Hospital San Angel Inn,19.340353,-99.199841,hospital
3,node/739035810,Clínica Hospital 8 IMSS,19.336612,-99.200969,hospital
4,way/883666969,Hospital General Dr. Fernando Quiroz Gutiérrez...,19.394387,-99.194223,hospital


In [24]:
#Cargamos el archivo json de todas las vialidades en Álvaro Obregón
vialidades = gpd.read_file("vialidades.geojson")

vias = vialidades[vialidades["highway"].notna()].copy()

In [27]:
#Creamos un csv de las posibles rutas viales dados los nodos del archivo original y nos quedamos con solo 3 para cada vialidad registrada
#Eliminamos la mayoría de los nodos pues excede el propósito del proyecto original, rescatamos los nodos del inicio medio y final de cada vialidad

def distancia_km(lat1, lon1, lat2, lon2):
    R = 6371 #Radio de la tierra en KM
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    #fórmula de Haversine. Sirve para medir distancias sobre una esfera, no en línea recta plana.
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c #Devuelve la distancia en kilómetros.

nodos = []
rutas = []

for i, row in vias.iterrows():
    coords = list(row.geometry.coords)

    if len(coords) < 2:
        continue

    indices = np.linspace(0, len(coords)-1, 3, dtype=int) #Escoge 3 posiciones dentro de la calle: inicio, punto medio y final.

    ids_via = []

    for k, idx in enumerate(indices):
        lon, lat = coords[idx]

        id_nodo = f"V{i}_{k}" 

        nodos.append({
            "id_nodo": id_nodo,
            "nombre_nodo": row.get("name", "vialidad_sin_nombre"), #Guarda el nombre de la calle. Si no tiene nombre, pone "vialidad_sin_nombre".
            "tipo": "vial",
            "lat": lat,
            "lon": lon,
            "highway": row.get("highway")
        })

        ids_via.append(id_nodo)

    for a, b in zip(ids_via, ids_via[1:]):
        nodo_a = nodos[-3 + ids_via.index(a)]
        nodo_b = nodos[-3 + ids_via.index(b)]
        #Calculamos la distancia en kilómetros entre los dos nodos.
        d = distancia_km( 
            nodo_a["lat"], nodo_a["lon"],
            nodo_b["lat"], nodo_b["lon"]
        )

        rutas.append({
            "origen": a,
            "destino": b,
            "distancia_km": round(d, 3),
            "factor_trafico": 1.0,
            "peso": round(d, 3)
        })

nodos_viales = pd.DataFrame(nodos) #puntos en el mapa
rutas_viales = pd.DataFrame(rutas) #conexiones entre esos puntos en el mapa



In [29]:
nodos_viales.to_csv("nodos_viales.csv", index=False)
rutas_viales.to_csv("rutas_viales.csv", index=False)


In [31]:
#A partir de los nodos que tenemos simulamos un csv de emergencias 
import pandas as pd

nodos = pd.read_csv("nodos_viales.csv")

# Elegir algunos nodos aleatorios como lugares de emergencia
nodos_emergencia = nodos.sample(n=9, random_state=42).copy()

especialidades = [
    "urgencias", "trauma", "cardiologia",
    "urgencias", "trauma", "cardiologia",
    "urgencias", "trauma", "cardiologia"
]

prioridades = [
    "ROJO", "AMARILLO", "VERDE",
    "ROJO", "VERDE", "AMARILLO",
    "AMARILLO", "ROJO", "VERDE"
]

# Crear emergencias
emergencias = pd.DataFrame({
    "id_emergencia": [f"E{i+1:02d}" for i in range(len(nodos_emergencia))], 
    "id_nodo_origen": nodos_emergencia["id_nodo"].values,
    "nombre_origen": nodos_emergencia["nombre_nodo"].values,
    "lat": nodos_emergencia["lat"].values,
    "lon": nodos_emergencia["lon"].values,
    "especialidad_requerida": especialidades,
    "prioridad": prioridades
})


emergencias

,id_emergencia,id_nodo_origen,nombre_origen,lat,lon,especialidad_requerida,prioridad
0,E01,V8_0,Avenida Jalisco,19.397430,-99.190014,urgencias,ROJO
1,E02,V2_0,Avenida San Jerónimo,19.332083,-99.207409,trauma,AMARILLO
2,E03,V31_0,Boulevard Reforma,19.384688,-99.244757,cardiologia,VERDE
3,E04,V36_1,Carretera México - Toluca (Cuota),19.373793,-99.263271,urgencias,ROJO
4,E05,V34_2,NaN,19.380395,-99.249598,trauma,VERDE
5,E06,V57_1,Anillo Periférico,19.356580,-99.198381,cardiologia,AMARILLO
6,E07,V77_2,Camino al Desierto de los Leones,19.347076,-99.202981,urgencias,AMARILLO
7,E08,V28_2,Carretera México - Toluca (Cuota),19.383038,-99.247418,trauma,ROJO
8,E09,V3_0,Avenida Paseo del Pedregal,19.308993,-99.204342,cardiologia,VERDE


In [33]:
emergencias.to_csv("emergencias.csv", index=False)
emergencias.to_excel("emergencias.xlsx", index=False)